# 09b — Enmiendas al plan de evaluación del test (v2)

**Qué hacemos acá.** Antes de abrir el test, se enmienda el plan congelado (v1): (1) se entrenan y congelan versiones de los modelos para la población de sensibilidad (120°), con los mismos hiperparámetros finales del 09; (2) se declara la comparación principal; (3) se separa el análisis de "qué baseline habría sido el mejor en test";
(4) se fijan las reglas del punto A y B; (5) calibración, (6) influencia de segmentos y (7) equidad descriptiva en test; (8) tabla de diferencias con train; (9) se congela el plan v2 y se conserva el v1.

**Reglas.** No se carga `test.parquet` ni ningún dato de test; solo se usan los conteos por segmento ya escritos (`07_conteos_test.csv`). Solo train y folds existentes. No se reejecuta el 09 y los archivos de la población principal no se tocan (se verifica su sha256 contra el v1).

## Lecturas tomadas (decisiones sin consulta)

1. **Modelo de la comparación principal:** se lee de `hiperparametros_B.json` (campo `recomendado_modelo`, escrito por el 09 con la prueba de elección); no se vuelve a elegir. El baseline principal es la regla por puntos top (`mejor_baseline_simple` del mismo archivo, elegido en train).
2. **Umbrales de los modelos de 120°:** se fijan con predicciones internas fuera de fold (GroupKFold de 3 por segmento) de toda la población de 120° de train, con los hiperparámetros finales del 09; para el modelo mínimo y las reglas, con sus puntajes sobre esa población (igual que el 09).
3. **Modelos de 120° y archivo de hiperparámetros:** además de los `*_B120.joblib` se guarda `hiperparametros_B120.json` (mismos hiperparámetros, umbrales nuevos) y se registran sus sha256.
4. **Tabla de referencia de train (sección 8 del plan):** se calcula con las predicciones fuera de fold del 09 (`oof_B.parquet`) y los umbrales por fold del 09, que se reconstruyen con la caché de la búsqueda anidada (solo train).
5. **Segmentos grandes:** los dos segmentos pedidos se identifican por el prefijo de 12 caracteres que usa el archivo de conteos.
6. **Contradicciones con el 09 (van al log):** el 09 aplicaba "mejor baseline simple de la población evaluada" también a la comparación; el plan v2 fija el baseline principal (regla por puntos top, elegido en train) y deja el mejor baseline de test como sensibilidad separada. Es un cambio de plan, no una contradicción de resultados.

In [1]:
from pathlib import Path
import datetime
import hashlib
import json
import shutil
import time

import joblib
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.compose import make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

SEMILLA = 42
pd.set_option("display.width", 170)
try:
    RAIZ = Path(context.project_path)
except NameError:
    RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "conf").exists() and (p / "data").exists())
RUTA_IN, RUTA_MOD, RUTA_REP, RUTA_OUT, RUTA_DOCS = (RAIZ / "data" / "05_model_input", RAIZ / "data" / "06_models", RAIZ / "data" / "08_reporting", RAIZ / "data" / "07_model_output", RAIZ / "docs")
RUTA_OUT.mkdir(exist_ok=True)


def pct(x, d=1):
    return f"{x * 100:.{d}f} %".replace(".", ",")


def n3(x):
    return f"{x:.3f}".replace(".", ",")


def md_tabla(df, indice=True):
    d = df.reset_index() if indice else df
    cab = "| " + " | ".join(map(str, d.columns)) + " |\n|" + "---|" * len(d.columns) + "\n"
    return cab + "\n".join("| " + " | ".join(str(v) for v in fila) + " |" for fila in d.values)


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


cfgB = json.loads((RUTA_IN / "config_features_B.json").read_text(encoding="utf-8"))
cfg06 = json.loads((RUTA_IN / "config_features.json").read_text(encoding="utf-8"))
SEGS_TEST = set(cfg06["segmentos_test"])           # solo ids, para los asserts
hj = json.loads((RUTA_MOD / "hiperparametros_B.json").read_text(encoding="utf-8"))
AZ_MAX, UMBRAL_GEOM = cfgB["AZ_MAX"], 105
VARS, GRUPO, TARGET = cfgB["recomendado_09"]["variables"], cfgB["grupo"], "sin_camara"
VARS_MIN = ["num_lidar_points", "num_top_lidar_points", "azimut_abs_deg"]
N_REGLA, N_TOP, N_GEOM, N_MIN = "Regla: menos puntos LiDAR", "Regla: menos puntos top", "Regla geometrica (azimut > 105 primero)", "Modelo minimo (LR 3 variables)"
NOMBRE_MOD = {"LR": "Regresion logistica", "HGB": "HGB", "RF": "Random Forest"}
print("Raiz:", RAIZ, "| AZ_MAX:", AZ_MAX, "| modelo de la comparacion principal (leido del 09):", hj["recomendado_modelo"], "| mejor baseline simple (train):", hj["mejor_baseline_simple"])

Raiz: C:\Users\maico\waymo-evaluacion | AZ_MAX: 100 | modelo de la comparacion principal (leido del 09): Regresion logistica | mejor baseline simple (train): Regla: menos puntos top


## 1. Conservar el plan v1 y verificar que la población principal no cambió

In [2]:
f_json, f_md = RUTA_MOD / "plan_evaluacion_test.json", RUTA_DOCS / "plan_evaluacion_test.md"
v1_json, v1_md = RUTA_MOD / "plan_evaluacion_test_v1.json", RUTA_DOCS / "plan_evaluacion_test_v1.md"
if not v1_json.exists():
    shutil.copy2(f_json, v1_json)
    shutil.copy2(f_md, v1_md)
    print("Plan v1 conservado como plan_evaluacion_test_v1.json / .md")
plan_v1 = json.loads(v1_json.read_text(encoding="utf-8"))
assert "v2" not in str(plan_v1.get("version", "")), "el archivo v1 ya es una v2"
archivos_ppal = {"hiperparametros_B.json": RUTA_MOD / "hiperparametros_B.json", "config_features_B.json": RUTA_IN / "config_features_B.json", "calibrador_B.joblib": RUTA_MOD / "calibrador_B.joblib",
                 "lr_minimo_B.joblib": RUTA_MOD / "lr_minimo_B.joblib", "lr_B.joblib": RUTA_MOD / "lr_B.joblib", "hgb_B.joblib": RUTA_MOD / "hgb_B.joblib", "rf_B.joblib": RUTA_MOD / "rf_B.joblib"}
sha_ahora = {k: sha(v) for k, v in archivos_ppal.items()}
sha_v1 = plan_v1["sha256"]["archivos"]
comp = pd.DataFrame({"sha256 en el plan v1": pd.Series(sha_v1), "sha256 ahora": pd.Series(sha_ahora)})
comp["sin cambios"] = comp["sha256 en el plan v1"] == comp["sha256 ahora"]
print(comp.assign(**{"sha256 en el plan v1": comp["sha256 en el plan v1"].str[:16] + "...", "sha256 ahora": comp["sha256 ahora"].str[:16] + "..."}).to_string())
assert comp["sin cambios"].all(), "algun archivo de la poblacion principal cambio respecto del plan v1"
print("\nVERIFICADO: los archivos de la poblacion principal no cambiaron respecto del plan v1.")

Plan v1 conservado como plan_evaluacion_test_v1.json / .md
                       sha256 en el plan v1         sha256 ahora  sin cambios
hiperparametros_B.json  dbf1c7d275dea216...  dbf1c7d275dea216...         True
config_features_B.json  c81ed082c26b3d86...  c81ed082c26b3d86...         True
calibrador_B.joblib     685e46723e88e80e...  685e46723e88e80e...         True
lr_minimo_B.joblib      d5dabd6c6b074a28...  d5dabd6c6b074a28...         True
lr_B.joblib             993c2f2bbace25f0...  993c2f2bbace25f0...         True
hgb_B.joblib            ac8328046919c4ad...  ac8328046919c4ad...         True
rf_B.joblib             b990648ec16b3b5d...  b990648ec16b3b5d...         True

VERIFICADO: los archivos de la poblacion principal no cambiaron respecto del plan v1.


## 2. Modelos congelados de la población de 120° (misma configuración final del 09, sin nueva búsqueda)

In [3]:
df_full = pd.read_parquet(RUTA_IN / "features_B.parquet")
assert set(df_full[GRUPO]).isdisjoint(SEGS_TEST), "hay segmentos de test en los datos"
assert len(df_full) == cfgB["n_filas"]
X120, y120, g120 = df_full[VARS].copy(), df_full[TARGET].values, df_full[GRUPO].values
az120 = df_full["azimut_abs_deg"].values
SKEW = [c for c in ["num_lidar_points", "num_top_lidar_points", "densidad_puntos", "volumen_caja"] if c in VARS]
RESTO = [c for c in VARS if c not in SKEW]
MONO_VARS = ["num_lidar_points", "num_top_lidar_points", "densidad_puntos"]


def prep_lr():
    return make_column_transformer((make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()), SKEW), (StandardScaler(), RESTO), verbose_feature_names_out=False)


def hgb_kwargs(cfg):
    kw = {k: v for k, v in cfg.items() if k != "mono"}
    if cfg.get("mono"):
        kw["monotonic_cst"] = {v: (-1 if v in MONO_VARS else 0) for v in VARS}
    return kw


def crear(nombre, cfg):
    if nombre == "LR":
        return make_pipeline(prep_lr(), LogisticRegression(C=cfg["C"], class_weight=cfg["class_weight"], max_iter=3000))
    if nombre == "HGB":
        return HistGradientBoostingClassifier(early_stopping=False, random_state=SEMILLA, **hgb_kwargs(cfg))
    return RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEMILLA, **cfg)


def crear_min():
    return make_pipeline(make_column_transformer((make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()), ["num_lidar_points", "num_top_lidar_points"]),
                                                  (StandardScaler(), ["azimut_abs_deg"]), verbose_feature_names_out=False), LogisticRegression(C=1.0, max_iter=2000))


def pred(m, Xd):
    return m.predict_proba(Xd)[:, 1]


def umbral_recall(s, yy, r):
    o = np.argsort(-s, kind="stable")
    return s[o][int(np.searchsorted(np.cumsum(yy[o]) / yy.sum(), r))]


def umbral_f1(s, yy):
    o = np.argsort(-s, kind="stable")
    ys = yy[o]
    tp = np.cumsum(ys)
    p_, r_ = tp / np.arange(1, len(ys) + 1), tp / ys.sum()
    return s[o][int(np.argmax(2 * p_ * r_ / (p_ + r_ + 1e-12)))]


def umbrales(s, yy):
    return {"A": float(umbral_recall(s, yy, 0.80)), "B": float(np.quantile(s, 0.80)), "F1": float(umbral_f1(s, yy))}


def oof_interno(nombre, cfg, X, yy, g):
    o = np.zeros(len(yy))
    for a, b in GroupKFold(n_splits=3).split(X, yy, g):
        o[b] = pred(crear(nombre, cfg).fit(X.iloc[a], yy[a]), X.iloc[b])
    return o


t0 = time.time()
B120 = {}
for m in ("LR", "HGB", "RF"):
    cfg = hj["final"][m]["cfg"]
    mod = crear(m, cfg).fit(X120, y120)
    inner = oof_interno(m, cfg, X120, y120, g120)
    B120[m] = {"cfg": cfg, "modelo": mod, "umbrales": umbrales(inner, y120)}
min120 = crear_min().fit(df_full[VARS_MIN], y120)
s_min120 = pred(min120, df_full[VARS_MIN])
s_reglas120 = {N_REGLA: -df_full["num_lidar_points"].values.astype(float), N_TOP: -df_full["num_top_lidar_points"].values.astype(float)}
s_reglas120[N_GEOM] = np.where(az120 > UMBRAL_GEOM, 1e6, 0.0) + s_reglas120[N_REGLA]
UMB_REGLAS_120 = {**{n: umbrales(s, y120) for n, s in s_reglas120.items()}, N_MIN: umbrales(s_min120, y120)}
print(f"Modelos de 120 grados ajustados en {time.time() - t0:.0f} s sobre {len(df_full):,} filas ({df_full[GRUPO].nunique()} segmentos de train).")
print(pd.DataFrame({NOMBRE_MOD[m]: {"configuracion": str(d["cfg"]), **{f"umbral {p}": round(v, 4) for p, v in d["umbrales"].items()}} for m, d in B120.items()}).T.to_string())
print("Umbrales de reglas y modelo minimo (120):", {n: {p: round(v, 3) for p, v in u.items()} for n, u in UMB_REGLAS_120.items()})

ARCH120 = {"LR": "lr_B120.joblib", "HGB": "hgb_B120.joblib", "RF": "rf_B120.joblib"}
for m, a in ARCH120.items():
    joblib.dump(B120[m]["modelo"], RUTA_MOD / a, compress=3 if m == "RF" else 0)
joblib.dump(min120, RUTA_MOD / "lr_minimo_B120.joblib")
hiper120 = {"poblacion": f"toda B de train (|azimut| <= 120), {len(df_full)} filas", "nota": "Mismos hiperparametros finales del 09 (sin nueva busqueda); umbrales fijados con predicciones internas fuera de fold (GroupKFold de 3 por segmento) de esta poblacion. Ningun dato de test.",
            "final": {m: {"cfg": d["cfg"], "umbrales_con_train_120": d["umbrales"]} for m, d in B120.items()}, "baselines_umbrales_con_train_120": UMB_REGLAS_120, "variables": VARS, "modelo_minimo_variables": VARS_MIN}
(RUTA_MOD / "hiperparametros_B120.json").write_text(json.dumps(hiper120, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

# comprobacion: cada modelo cargado reproduce sus predicciones en una muestra de train
idx = np.random.default_rng(SEMILLA).choice(len(df_full), 3000, replace=False)
verif120 = {}
for m, a in ARCH120.items():
    verif120[a] = float(np.max(np.abs(pred(joblib.load(RUTA_MOD / a), X120.iloc[idx]) - pred(B120[m]["modelo"], X120.iloc[idx]))))
verif120["lr_minimo_B120.joblib"] = float(np.max(np.abs(pred(joblib.load(RUTA_MOD / "lr_minimo_B120.joblib"), df_full.iloc[idx][VARS_MIN]) - pred(min120, df_full.iloc[idx][VARS_MIN]))))
assert all(v < 1e-9 for v in verif120.values()), verif120
h_leido = json.loads((RUTA_MOD / "hiperparametros_B120.json").read_text(encoding="utf-8"))
for m in B120:
    assert all(np.isclose(h_leido["final"][m]["umbrales_con_train_120"][p], B120[m]["umbrales"][p]) for p in "A B F1".split())
sha120 = {a: sha(RUTA_MOD / a) for a in [*ARCH120.values(), "lr_minimo_B120.joblib", "hiperparametros_B120.json"]}
print("Verificacion (diferencia maxima de predicciones al recargar):", {k: f"{v:.1e}" for k, v in verif120.items()})
print("sha256 de los archivos de 120 grados:", {k: v[:16] + "..." for k, v in sha120.items()})

Modelos de 120 grados ajustados en 9 s sobre 68,555 filas (33 segmentos de train).
                                                                                                                                                                                                           configuracion umbral A umbral B umbral F1
Regresion logistica                                                                                                                                                                   {'C': 0.001, 'class_weight': None}   0.1623   0.3663    0.3162
HGB                  {'class_weight': None, 'l2_regularization': 9.6098129470364, 'learning_rate': 0.030511586180091473, 'max_depth': None, 'max_leaf_nodes': 4, 'min_samples_leaf': 100, 'mono': False, 'max_iter': 90}   0.1359   0.3712    0.3702
Random Forest                                                                                                                                          {'min_samples_leaf': 200, 'max_

Verificacion (diferencia maxima de predicciones al recargar): {'lr_B120.joblib': '0.0e+00', 'hgb_B120.joblib': '0.0e+00', 'rf_B120.joblib': '3.3e-16', 'lr_minimo_B120.joblib': '0.0e+00'}
sha256 de los archivos de 120 grados: {'lr_B120.joblib': 'f4c80b104d4a157e...', 'hgb_B120.joblib': '37ea6400a7b32816...', 'rf_B120.joblib': 'cd9a0fc66f8975a3...', 'lr_minimo_B120.joblib': '545bfac46ae29424...', 'hiperparametros_B120.json': '421ed63cf989c8d2...'}


## 3. Referencia de train para el plan: métricas fuera de fold del 09 (población principal)

Se reconstruyen con `oof_B.parquet` y, para los umbrales por fold, con la caché de la búsqueda anidada (solo train). Sirve de columna "train" de la tabla de diferencias con train (sección 8 del plan).

In [4]:
oof = pd.read_parquet(RUTA_OUT / "oof_B.parquet")
oof = oof[oof["poblacion"] == "principal"].reset_index(drop=True)
dfp = df_full[df_full["en_zona_principal"]].reset_index(drop=True)
assert len(oof) == len(dfp) and (oof["key.laser_object_id"].values == dfp["key.laser_object_id"].values).all()
assert set(oof[GRUPO]).isdisjoint(SEGS_TEST)
y = oof["sin_camara"].values
folds = oof["fold"].values.astype(int)
cache = joblib.load(RUTA_OUT / "_cache_09_busqueda.joblib")
busq = cache["busq"]
SC = {N_REGLA: oof["score_regla_puntos"].values, N_TOP: oof["score_regla_top"].values, N_GEOM: oof["score_regla_geometrica"].values, N_MIN: oof["score_modelo_minimo"].values,
      NOMBRE_MOD["LR"]: oof["score_lr"].values, NOMBRE_MOD["HGB"]: oof["score_hgb"].values, NOMBRE_MOD["RF"]: oof["score_rf"].values}
tb = np.random.default_rng(SEMILLA).random(len(y))


def f80(yy, s):
    o = np.lexsort((tb[: len(yy)], -s))
    cum = np.cumsum(yy[o]) / yy.sum()
    return (np.searchsorted(cum, 0.8) + 1) / len(cum)


# umbrales por fold (los mismos del 09): modelos con predicciones internas; reglas con el puntaje del entrenamiento; modelo minimo ajustado dentro del fold
FLAGS = {n: {"A": np.zeros(len(y), bool), "B": np.zeros(len(y), bool)} for n in SC}
for k in sorted(np.unique(folds)):
    tr, va = folds != k, folds == k
    ref_tr = {N_REGLA: SC[N_REGLA][tr], N_TOP: SC[N_TOP][tr], N_GEOM: SC[N_GEOM][tr]}
    mm = crear_min().fit(dfp.loc[tr, VARS_MIN], y[tr])
    ref_tr[N_MIN] = pred(mm, dfp.loc[tr, VARS_MIN])
    for m in ("LR", "HGB", "RF"):
        ref_tr[NOMBRE_MOD[m]] = busq[m][k]["inner_oof"]
    for n in SC:
        u = umbrales(ref_tr[n], y[tr])
        FLAGS[n]["A"][va], FLAGS[n]["B"][va] = SC[n][va] >= u["A"], SC[n][va] >= u["B"]
filas = []
for n, s in SC.items():
    fa, fb = FLAGS[n]["A"], FLAGS[n]["B"]
    tpa, tpb = (fa & (y == 1)).sum(), (fb & (y == 1)).sum()
    filas.append({"metodo": n, "AUC (OOF agrupado)": roc_auc_score(y, s), "PR-AUC": average_precision_score(y, s), "Fraccion para 80 %": f80(y, s), "recall A": tpa / y.sum(), "precision A": tpa / max(fa.sum(), 1), "% revisado A": fa.mean(),
                  "recall B": tpb / y.sum(), "% revisado B": fb.mean()})
ref_train = pd.DataFrame(filas).set_index("metodo")
print(ref_train.round(3).to_string())

                                         AUC (OOF agrupado)  PR-AUC  Fraccion para 80 %  recall A  precision A  % revisado A  recall B  % revisado B
metodo                                                                                                                                              
Regla: menos puntos LiDAR                             0.858   0.698               0.385     0.804        0.433         0.385     0.660         0.209
Regla: menos puntos top                               0.862   0.704               0.378     0.806        0.441         0.379     0.665         0.210
Regla geometrica (azimut > 105 primero)               0.858   0.698               0.385     0.804        0.433         0.385     0.660         0.209
Modelo minimo (LR 3 variables)                        0.855   0.702               0.384     0.799        0.455         0.364     0.654         0.203
Regresion logistica                                   0.825   0.657               0.437     0.810        0

## 4. Plan v2: construcción y congelamiento

In [5]:
# ---- conteos conocidos del test (solo conteos por segmento)
conteos = pd.read_csv(RUTA_REP / "07_conteos_test.csv", dtype={"segment_id": str})
assert set(conteos.columns) <= {"segment_id", "location", "time_of_day", "filas_B_120", "sin_camara_120", "filas_B_principal", "sin_camara_principal", "filas_en_franja", "sin_en_franja"} and len(conteos) == len(SEGS_TEST)
ids_grandes = ["257766998801", "221704303323"]
assert all(i in set(conteos["segment_id"].astype(str)) for i in ids_grandes), "los segmentos grandes no estan en los conteos"
evaluables = conteos[conteos["sin_camara_principal"] >= 50]["segment_id"].astype(str).tolist()
momento = conteos.groupby("time_of_day")["segment_id"].nunique().to_dict()
ubic = conteos.groupby("location")["segment_id"].nunique().to_dict()
tot = {c: int(conteos[c].sum()) for c in conteos.columns if c.startswith(("filas", "sin"))}
conteos["filas_B_principal"] = conteos["filas_B_principal"].astype(int)
grandes = conteos[conteos["segment_id"].astype(str).isin(ids_grandes)][["segment_id", "filas_B_principal", "sin_camara_principal"]]

MODELO_PPAL = hj["recomendado_modelo"]
BASE_PPAL = hj["mejor_baseline_simple"]
assert BASE_PPAL == N_TOP, f"el baseline principal elegido en train es {BASE_PPAL}"
ARCH_PPAL = {"Regresion logistica": "lr_B.joblib", "HGB": "hgb_B.joblib", "Random Forest": "rf_B.joblib"}[MODELO_PPAL]
cal_train = json.loads(json.dumps(plan_v1["calibrador"]))
cr = plan_v1["criterios"]["parametros"]
ahora = datetime.datetime.now().astimezone().isoformat(timespec="seconds")

cambios = [
    ("Modelos de la sensibilidad a 120°", "En v1 la sensibilidad usaba los modelos entrenados en la población principal. Ahora hay modelos congelados entrenados sobre toda la población de 120° de train (`*_B120.joblib`), con los mismos hiperparámetros finales del 09 (sin nueva búsqueda) y umbrales fijados con predicciones internas fuera de fold de esa población. Motivo: que la sensibilidad compare modelos entrenados en la población que evalúa."),
    ("Comparación principal pre-declarada", f"Se declara una sola comparación principal: población principal (|azimut| ≤ {AZ_MAX}°), modelo {MODELO_PPAL} (el mejor modelo aprendido según la prueba de elección del 09, leído de `hiperparametros_B.json`) frente a la {BASE_PPAL} (el mejor baseline simple elegido en train), con el criterio 'vale la pena' del 09. Todo lo demás es secundario. Motivo: evitar elegir la comparación después de ver el test."),
    ("El mejor baseline del test sale del criterio principal", "En v1 el mejor baseline simple se elegía por AUC del test. Ahora el criterio principal usa el baseline elegido en train, y 'qué baseline habría sido el mejor en test' es un análisis de sensibilidad separado. Motivo: no usar el test para elegir contra qué se compara."),
    ("Punto A sostenido", "El punto A se considera sostenido si el recall puntual en test es ≥ 0,75; se reporta además el % de frames revisados frente al de train. El punto B se reporta sin tolerancia. No se recalibra ni se mueve ningún umbral. Motivo: fijar de antemano qué se entiende por sostenido."),
    ("Calibración en test", "Descriptiva: ECE, Brier y curva de confiabilidad del mejor modelo aprendido con el calibrador congelado. Motivo: saber si las probabilidades publicadas conservarían su significado."),
    ("Análisis de influencia", "Se repite la comparación principal excluyendo cada uno de los dos segmentos más grandes, ambos, y leave-one-segment-out (8 resultados), con el rango. Motivo: con 8 segmentos y dos que concentran la mayor parte de las filas, el resultado puede depender de uno solo."),
    ("Equidad descriptiva", "Por ubicación, momento del día y clase, con número de segmentos y filas, recall al punto A y AUC solo con ≥ 200 filas por clase, sin pruebas de hipótesis. Motivo: describir sin sobreinterpretar grupos de un solo segmento."),
    ("Tabla de diferencias con train", "Para cada método: AUC, Fracción para 80 %, recall y % revisado en el punto A, en test frente al OOF de train. Motivo: ver qué cambia entre train y test y por qué."),
    ("Congelamiento v2", "El plan v1 se conserva (`plan_evaluacion_test_v1.*`); este es el v2, con nuevos sha256 (incluidos los `*_B120.joblib`) y nuevo sello de fecha y hora."),
]

tabla_ref = ref_train.copy()
tabla_ref_fmt = tabla_ref.copy()
for c in tabla_ref_fmt.columns:
    tabla_ref_fmt[c] = tabla_ref_fmt[c].map(lambda v: f"{v:.3f}")

plan = dict(plan_v1)
plan.update({
    "version": "v2",
    "congelado": ahora,
    "estado": "Plan v2 congelado antes de abrir el test (enmienda del v1, conservado como plan_evaluacion_test_v1). El test no se usa para elegir modelos, umbrales, hiperparametros, la comparacion principal ni el baseline principal.",
    "cambios_v1_v2": [{"cambio": a, "por_que": b} for a, b in cambios],
    "comparacion_principal": {
        "poblacion": f"principal: peatones y ciclistas con |azimut| <= {AZ_MAX}",
        "modelo": MODELO_PPAL, "archivo_modelo": f"data/06_models/{ARCH_PPAL}", "baseline": BASE_PPAL, "puntaje_baseline": "-num_top_lidar_points",
        "criterio_vale_la_pena": "la Fraccion para 80 % del modelo baja >= 5 puntos frente al baseline, con IC 95 % pareado por bootstrap de segmentos de test que excluya 0, y el AUC sube >= 0,03 (identico al criterio del 09)",
        "fuente_del_modelo": "campo recomendado_modelo de data/06_models/hiperparametros_B.json (resultado de la prueba de eleccion del 09; no se elige de nuevo)",
        "todo_lo_demas": "secundario: otros modelos, la poblacion de 120 grados y otras metricas se reportan como secundarios"},
    "sensibilidad_mejor_baseline_en_test": "analisis separado: que baseline simple habria sido el mejor en test (mayor AUC en cada poblacion) y como cambia la comparacion; no entra en el criterio principal",
    "punto_A_B": {"A": "recall objetivo 0,80 fijado con train; se considera sostenido en test si el recall puntual es >= 0,75; se reporta tambien el % de frames revisados frente al de train",
                  "B": "revisar el 20 % mejor puntuado segun el corte fijado con train; se reporta sin tolerancia", "regla": "no se recalibra ni se mueve ningun umbral"},
    "calibracion_en_test": {"modelo": MODELO_PPAL, "calibrador": cal_train, "metricas": "ECE (10 bins por cuantiles), Brier y curva de confiabilidad, con el calibrador congelado (la version que gano en train); descriptivo"},
    "analisis_de_influencia": {"comparacion": "AUC y Fraccion para 80 % del modelo menos el baseline (poblacion principal)", "segmentos_grandes": ids_grandes, "variantes": ["excluir cada segmento grande", "excluir ambos", "leave-one-segment-out (8 resultados)"],
                               "reporte": "rango de los resultados", "metricas_por_segmento": f"solo en segmentos con >= 50 positivos en la poblacion principal: {evaluables}; el resto cuenta solo en los agregados"},
    "equidad_descriptiva": {"grupos": ["ubicacion", "momento del dia", "clase"], "reporta": "numero de segmentos y filas por grupo, recall al punto A; AUC solo si el grupo tiene >= 200 filas por clase", "sin_pruebas_de_hipotesis": True,
                            "aclaracion": f"Night y Dawn/Dusk tienen un solo segmento cada uno en el test (segmentos por momento del dia: {momento}); por ubicacion: {ubic}. La equidad por momento del dia es descriptiva de un segmento por grupo."},
    "diferencias_con_train": {"referencia_train_oof_principal": {n: {k: float(v) for k, v in r.items()} for n, r in ref_train.iterrows()},
                              "nota": "En test cada modelo es unico y congelado (no hay mezcla de escalas entre folds), por lo que sus metricas agrupadas no sufren el sesgo del OOF agrupado de train (cinco modelos distintos, uno por fold). Esto puede hacer que los modelos se vean mejor en test que en el 09; la tabla de diferencias debe leerse con esa advertencia."},
    "modelos_sensibilidad_120": {"archivos": {a: f"data/06_models/{a}" for a in [*ARCH120.values(), "lr_minimo_B120.joblib"]}, "hiperparametros": "data/06_models/hiperparametros_B120.json",
                                 "umbrales_con_train_120": {NOMBRE_MOD[m]: d["umbrales"] for m, d in B120.items()}, "umbrales_reglas_y_modelo_minimo_120": UMB_REGLAS_120,
                                 "nota": "mismos hiperparametros finales del 09 (sin nueva busqueda); umbrales con predicciones internas fuera de fold de toda la poblacion de 120 de train"},
    "composicion_conocida_del_test": {"segmentos": int(len(conteos)), "totales": tot, "por_segmento": conteos.to_dict(orient="records"), "segmentos_con_50_positivos_principal": evaluables, "fuente": "data/08_reporting/07_conteos_test.csv (solo conteos, escrito por el notebook 07)"},
    "reglas": ["El test no se usa para elegir modelos, umbrales, hiperparametros, comparacion principal ni baseline principal.", "Los modelos, umbrales y calibrador se aplican tal como estan congelados, sin reajuste.",
               "Cualquier desviacion de este plan se documenta en el notebook 10 con su motivo.", "Las poblaciones, metricas y criterios son los de este plan; no se agregan metricas despues de ver el test.",
               "Los analisis secundarios y de sensibilidad se reportan como tales y no cambian la conclusion principal."],
})
# metodos: se agregan los modelos de 120 grados
metodos = [m_ for m_ in plan_v1["metodos"]]
for m, a in ARCH120.items():
    metodos.append({"nombre": f"{NOMBRE_MOD[m]} (120 grados)", "tipo": "modelo (sensibilidad)", "puntaje": "probabilidad predicha de sin_camara", "archivo": f"data/06_models/{a}", "sha256": sha120[a], "variables": VARS,
                    "configuracion_final": B120[m]["cfg"], "umbrales_con_train": B120[m]["umbrales"], "nota_umbral": "umbral fijado con predicciones internas fuera de fold de toda la poblacion de 120 de train"})
metodos.append({"nombre": "Modelo minimo (120 grados)", "tipo": "modelo minimo (sensibilidad)", "puntaje": "probabilidad predicha", "archivo": "data/06_models/lr_minimo_B120.joblib", "sha256": sha120["lr_minimo_B120.joblib"], "variables": VARS_MIN,
                "umbrales_con_train": UMB_REGLAS_120[N_MIN], "nota_umbral": "umbral con el puntaje sobre toda la poblacion de 120 de train"})
plan["metodos"] = metodos
plan["reglas_en_120"] = {n: {"puntaje": d, "umbrales_con_train_120": UMB_REGLAS_120[n]} for n, d in [(N_REGLA, "-num_lidar_points"), (N_TOP, "-num_top_lidar_points"), (N_GEOM, "1e6*(azimut_abs_deg > 105) - num_lidar_points")]}
sha_todos = {**sha_ahora, **sha120}
plan["sha256"] = {"archivos": sha_todos, "features_B.parquet": sha(RUTA_IN / "features_B.parquet"), "plan_v1_json": sha(v1_json)}
plan["verificacion_con_train"] = {"poblacion_principal_sin_cambios_respecto_de_v1": True, "diferencia_maxima_modelos_120": verif120, "muestra": 3000}
plan["contradicciones_con_el_09"] = ["El 09 comparaba contra el mejor baseline simple de la poblacion evaluada; el plan v2 fija el baseline principal (regla por puntos top, elegido en train) y deja el mejor baseline de test como sensibilidad separada. Es un cambio de plan, no un cambio de resultados."]
f_json.write_text(json.dumps(plan, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("Plan v2 escrito en", f_json)

Plan v2 escrito en C:\Users\maico\waymo-evaluacion\data\06_models\plan_evaluacion_test.json


In [6]:
# ---- documento .md del plan v2
tabla_met = pd.DataFrame([{"método": m_["nombre"], "tipo": m_["tipo"], "archivo": m_["archivo"] or "(regla, sin archivo)", "umbral A": f"{m_['umbrales_con_train']['A']:.4g}", "umbral B": f"{m_['umbrales_con_train']['B']:.4g}",
                           "umbral F1": f"{m_['umbrales_con_train']['F1']:.4g}", "sha256": (m_.get("sha256") or "-")[:16] + ("..." if m_.get("sha256") else "")} for m_ in plan["metodos"]])
tabla_reglas120 = pd.DataFrame([{"regla": n, "puntaje": d["puntaje"], **{f"umbral {p}": f"{v:.4g}" for p, v in d["umbrales_con_train_120"].items()}} for n, d in plan["reglas_en_120"].items()])
cp = conteos.copy()
cp["segment_id"] = cp["segment_id"].astype(str)
partes = [
    f"# Plan de evaluación del test (v2, congelado)\n\n**Congelado el {ahora}.** {plan['estado']}\n\nEl plan v1 se conserva en `plan_evaluacion_test_v1.md` y `data/06_models/plan_evaluacion_test_v1.json`.\n",
    "## Cambios entre v1 y v2 y por qué (todos hechos antes de abrir el test)\n\n" + "\n".join(f"{i + 1}. **{a}.** {b}" for i, (a, b) in enumerate(cambios)) + "\n\n" + "**Contradicciones con el 09:** " + plan["contradicciones_con_el_09"][0] + "\n",
    "## 1. Comparación principal (pre-declarada)\n\n" + "\n".join(f"- **{k.replace('_', ' ')}:** {v}" for k, v in plan["comparacion_principal"].items()) +
    "\n\nLa conclusión principal del notebook 10 sale únicamente de esta comparación. Los demás modelos, la población de 120° y las demás métricas se reportan como **secundarios**.\n",
    "## 2. Sensibilidad separada: qué baseline habría sido el mejor en test\n\n" + plan["sensibilidad_mejor_baseline_en_test"] + ".\n",
    "## 3. Puntos de operación\n\n" + "\n".join(f"- **{k}:** {v}" for k, v in plan["punto_A_B"].items()) + "\n",
    "## 4. Calibración en test (descriptiva)\n\n" + f"Modelo {MODELO_PPAL} con el calibrador congelado (`{cal_train['archivo']}`, versión que ganó en train: {cal_train['mejor_version_en_train']}): ECE (10 bins por cuantiles), Brier y curva de confiabilidad. Descriptivo, sin umbral de aceptación.\n",
    "## 5. Análisis de influencia\n\n" + f"- Comparación: {plan['analisis_de_influencia']['comparacion']}.\n- Segmentos más grandes: {', '.join(ids_grandes)} (filas y positivos en la población principal: " + "; ".join(f"{r.segment_id}: {r.filas_B_principal} y {r.sin_camara_principal}" for r in grandes.itertuples()) + ").\n" +
    "- Variantes: " + "; ".join(plan["analisis_de_influencia"]["variantes"]) + ".\n- Se reporta el rango de los resultados.\n- " + plan["analisis_de_influencia"]["metricas_por_segmento"] + ".\n",
    "## 6. Equidad descriptiva\n\n" + f"{plan['equidad_descriptiva']['reporta'].capitalize()}. Sin pruebas de hipótesis.\n\n**{plan['equidad_descriptiva']['aclaracion']}**\n",
    "## 7. Diferencias con train\n\nTabla a completar en el notebook 10: para cada método, AUC, Fracción para 80 %, recall y % revisado en el punto A, en test frente a la referencia de train (OOF de la población principal, abajo).\n\n" + md_tabla(tabla_ref_fmt) + "\n\n" + plan["diferencias_con_train"]["nota"] + "\n",
    "## 8. Métodos, archivos y umbrales\n\n" + md_tabla(tabla_met, indice=False) + "\n\nUmbrales de las reglas en la población de 120° (con train):\n\n" + md_tabla(tabla_reglas120, indice=False) +
    "\n\nLos umbrales de las reglas son sobre su puntaje; los de los modelos, sobre la probabilidad predicha. Los modelos de la población principal se entrenaron y fijaron sus umbrales en la población principal; los `(120 grados)`, en toda la población de 120°.\n",
    "## 9. Poblaciones, métricas, criterios y redacción\n\n" + f"- **Principal:** {plan['poblaciones']['principal']}.\n- **Sensibilidad:** {plan['poblaciones']['sensibilidad']}; se evalúan los modelos `(120 grados)` y las reglas.\n- **Métricas:** " + "; ".join(plan["metricas"]) + f".\n- **Intervalos:** {plan['intervalos']}.\n" +
    "\n**Criterios (idénticos a los del 09):**\n\n" + "\n".join(f"- **{k.replace('_', ' ')}:** {v}" for k, v in plan["criterios"].items() if isinstance(v, str)) + f"\n\n**Lectura complementaria:** {plan['lectura_complementaria']}\n\n**Cómo se redactará la conclusión en cada caso:**\n\n" +
    "\n".join(f"- **{k.replace('_', ' ')}:** {v}" for k, v in plan["redaccion_de_la_conclusion"].items()) + "\n",
    "## 10. Reglas\n\n" + "\n".join(f"- {t}" for t in plan["reglas"]) + "\n",
    "## 11. Composición conocida del test (solo conteos)\n\n" + md_tabla(cp, indice=False) + "\n\nTotales: " + ", ".join(f"{k} = {v:,}".replace(",", ".") for k, v in tot.items()) + f". Segmentos con ≥ 50 positivos en la población principal: {', '.join(evaluables)}.\n",
    "## 12. Sumas de verificación (sha256)\n\n" + "\n".join(f"- `{k}`: `{v}`" for k, v in sha_todos.items()) + f"\n- `features_B.parquet`: `{plan['sha256']['features_B.parquet']}`\n- `plan_evaluacion_test_v1.json`: `{plan['sha256']['plan_v1_json']}`\n",
]
(RUTA_DOCS / "plan_evaluacion_test.md").write_text("\n".join(partes), encoding="utf-8")
print(f"Plan v2 congelado el {ahora}")
print("Archivos de la poblacion principal sin cambios respecto de v1: True")
display(Markdown("\n".join(partes[:3])))

Plan v2 congelado el 2026-10-18T13:42:13-03:00
Archivos de la poblacion principal sin cambios respecto de v1: True


# Plan de evaluación del test (v2, congelado)

**Congelado el 2026-10-18T13:42:13-03:00.** Plan v2 congelado antes de abrir el test (enmienda del v1, conservado como plan_evaluacion_test_v1). El test no se usa para elegir modelos, umbrales, hiperparametros, la comparacion principal ni el baseline principal.

El plan v1 se conserva en `plan_evaluacion_test_v1.md` y `data/06_models/plan_evaluacion_test_v1.json`.

## Cambios entre v1 y v2 y por qué (todos hechos antes de abrir el test)

1. **Modelos de la sensibilidad a 120°.** En v1 la sensibilidad usaba los modelos entrenados en la población principal. Ahora hay modelos congelados entrenados sobre toda la población de 120° de train (`*_B120.joblib`), con los mismos hiperparámetros finales del 09 (sin nueva búsqueda) y umbrales fijados con predicciones internas fuera de fold de esa población. Motivo: que la sensibilidad compare modelos entrenados en la población que evalúa.
2. **Comparación principal pre-declarada.** Se declara una sola comparación principal: población principal (|azimut| ≤ 100°), modelo Regresion logistica (el mejor modelo aprendido según la prueba de elección del 09, leído de `hiperparametros_B.json`) frente a la Regla: menos puntos top (el mejor baseline simple elegido en train), con el criterio 'vale la pena' del 09. Todo lo demás es secundario. Motivo: evitar elegir la comparación después de ver el test.
3. **El mejor baseline del test sale del criterio principal.** En v1 el mejor baseline simple se elegía por AUC del test. Ahora el criterio principal usa el baseline elegido en train, y 'qué baseline habría sido el mejor en test' es un análisis de sensibilidad separado. Motivo: no usar el test para elegir contra qué se compara.
4. **Punto A sostenido.** El punto A se considera sostenido si el recall puntual en test es ≥ 0,75; se reporta además el % de frames revisados frente al de train. El punto B se reporta sin tolerancia. No se recalibra ni se mueve ningún umbral. Motivo: fijar de antemano qué se entiende por sostenido.
5. **Calibración en test.** Descriptiva: ECE, Brier y curva de confiabilidad del mejor modelo aprendido con el calibrador congelado. Motivo: saber si las probabilidades publicadas conservarían su significado.
6. **Análisis de influencia.** Se repite la comparación principal excluyendo cada uno de los dos segmentos más grandes, ambos, y leave-one-segment-out (8 resultados), con el rango. Motivo: con 8 segmentos y dos que concentran la mayor parte de las filas, el resultado puede depender de uno solo.
7. **Equidad descriptiva.** Por ubicación, momento del día y clase, con número de segmentos y filas, recall al punto A y AUC solo con ≥ 200 filas por clase, sin pruebas de hipótesis. Motivo: describir sin sobreinterpretar grupos de un solo segmento.
8. **Tabla de diferencias con train.** Para cada método: AUC, Fracción para 80 %, recall y % revisado en el punto A, en test frente al OOF de train. Motivo: ver qué cambia entre train y test y por qué.
9. **Congelamiento v2.** El plan v1 se conserva (`plan_evaluacion_test_v1.*`); este es el v2, con nuevos sha256 (incluidos los `*_B120.joblib`) y nuevo sello de fecha y hora.

**Contradicciones con el 09:** El 09 comparaba contra el mejor baseline simple de la poblacion evaluada; el plan v2 fija el baseline principal (regla por puntos top, elegido en train) y deja el mejor baseline de test como sensibilidad separada. Es un cambio de plan, no un cambio de resultados.

## 1. Comparación principal (pre-declarada)

- **poblacion:** principal: peatones y ciclistas con |azimut| <= 100
- **modelo:** Regresion logistica
- **archivo modelo:** data/06_models/lr_B.joblib
- **baseline:** Regla: menos puntos top
- **puntaje baseline:** -num_top_lidar_points
- **criterio vale la pena:** la Fraccion para 80 % del modelo baja >= 5 puntos frente al baseline, con IC 95 % pareado por bootstrap de segmentos de test que excluya 0, y el AUC sube >= 0,03 (identico al criterio del 09)
- **fuente del modelo:** campo recomendado_modelo de data/06_models/hiperparametros_B.json (resultado de la prueba de eleccion del 09; no se elige de nuevo)
- **todo lo demas:** secundario: otros modelos, la poblacion de 120 grados y otras metricas se reportan como secundarios

La conclusión principal del notebook 10 sale únicamente de esta comparación. Los demás modelos, la población de 120° y las demás métricas se reportan como **secundarios**.
